In [3]:
import pandas as pd
import numpy as np

### Load the data

In [4]:
cubo = pd.read_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\cross selling\data\cubo_1.xlsx", 
                    sheet_name = 'SKUs',
                    skiprows=4
                    )

In [ ]:
#ventas = 'C:\\Users\\fdavila\\OneDrive - Farmacorp S.A\\Escritorio\\cross selling\\data\\DATOS DE VENTA DE ENE-MAR 2025.xlsx'

In [ ]:
ventasune = pd.read_csv(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\cross selling\data\ventas_01_03-25.csv")
#ventasune = pd.read_csv(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\cross selling\data\DATOS DE VENTA DE ENE-MAR_ 2025 amarke.csv", sep=";")

In [4]:
ventasune['NUMERO_FACTURA'].nunique()

5163253

In [5]:
clusters = pd.read_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\data\clusters_bdgs.xlsx")

In [6]:
#convertimos fechas a formato datetime
ventasune['FECHA_FACT']= pd.to_datetime(ventasune['FECHA_FACT'],
                                           #format='%d/%m/%Y'
                                           )

In [8]:
ventasfactxdia = (
    ventasune
    .groupby('FECHA_FACT')['NUMERO_FACTURA']
    .nunique()
    .reset_index(name='FACTURAS_UNICAS')
)


In [9]:
ventasfactxdia

,FECHA_FACT,FACTURAS_UNICAS
0,2025-01-01,54545
1,2025-01-02,65059
2,2025-01-03,63503
3,2025-01-04,59396
4,2025-01-05,55910
...,...,...
85,2025-03-27,58353
86,2025-03-28,60505
87,2025-03-29,56075
88,2025-03-30,57274


Observamos que el dia que mas se han emitido facturas en FARMACORP fue el dia 05 de Marzo correspondiente al dia despues del feriado de Carnavales, pero las unidades vendidas no fueron las mas altas, esto podria significar que se emitieron muchas facturas con pocas unidades por factura.

In [10]:
#visualizamos las ventas por dia
ventasxdia = ventasune.FECHA_FACT.value_counts().reset_index()

In [11]:
ventasxdia

,FECHA_FACT,count
0,2025-03-01,133659
1,2025-02-28,128955
2,2025-01-02,128305
3,2025-03-05,127125
4,2025-03-31,126489
...,...,...
85,2025-01-30,101507
86,2025-03-03,98300
87,2025-01-29,97832
88,2025-03-04,94747


El dia que mas unidades se ha facturado en FARMACORP en el lapso de estos 3 mese fue el 01 de Marzo que corresponde al Sabado anterior al feriado de Carnavales. Sin embargo este dia no se han emitido tantas facturas como el 05 de Marzo, esto refleja que se vendieron mas unidades por factura.

In [12]:
#vamos a filtrar solo BODEGAS FARMACORP para este analisis
clusters_une = clusters[clusters['UNE']=='FARMACORP']
# clusters_une = clusters[clusters['UNE']=='AMARKET']

In [13]:
ventasxfact = ventasune.copy()

In [ ]:
#analizamos el dia com mas movimiento (SABADO antes de Carnavales)
#ventasxfact = ventasfarma[(ventasfarma['FECHA_FACT'] == '2025-03-01 00:00:00')]
#ventasxfact = ventasfarma[(ventasfarma['FECHA_FACT'] == '2025-03-05 00:00:00')]

In [14]:
#analizamos los dias de carnavales
ventasxfact = ventasxfact[(ventasxfact['FECHA_FACT'] >= '2025-02-01 00:00:00') & (ventasxfact['FECHA_FACT'] <= '2025-03-04 00:00:00')]

## CROSS SELLING X PERIODO

### Procesamos la data

In [15]:
cubo['COD_ARTICULO'] = cubo['COD_ARTICULO'].astype(str)
ventasxfact['COD_ARTICULO'] = ventasxfact['COD_ARTICULO'].astype(str)

In [16]:
# filtering only SKUs on FARMACORP
ventas_une = pd.merge(cubo,
                            ventasxfact,
                            on='COD_ARTICULO',
                            how='inner'
                            )

In [17]:
ventas_une['CAT 0'].value_counts()

CAT 0
ETICOS              1168365
OTC                  969520
CUIDADO PERSONAL     296809
BEBIDAS              282620
IMPULSO              231594
INSUMOS MEDICOS      215004
PERECEDEROS          157448
CUIDADO INFANTIL     103102
HOGAR                 55840
ABARROTES             32131
Name: count, dtype: int64

Las categorias que mas venden en FARMACORP son (Unidades Vendidas):
* ETICOS
* OTC
* CUIDADO PERSONAL
Las categorias que mas venden en AMARKET son (Unidades Vendidas):
* PERECEDEROS
* BEBIDAS
* IMPULSO

In [18]:
# Evaluamos OTC solo
#ventas_sample = ventas_une[
    #(ventas_une['CAT 0'].isin(['HOGAR'])) 
    #|  (ventas_amarket['CAT 1'].isin(['CERVEZAS', 'DESTILADOS-VINOS Y ESPUMANTES']))]
ventas_sample = ventas_une.copy()

In [19]:
ventas_sample

,CAT 0,CAT 1,CAT 2,CAT 3,CAT 4,COD_ARTICULO,ARTICULO,Precio Unitario FA,CR Unitario,FECHA_FACT,COD_BODEGA,NUMERO_FACTURA,UNIDADES,VENTA_NETA
0,ABARROTES,ABARROTES,ACEITES VEGETALES,ACEITES DE GIRASOL Y MAIZ,ACEITES DE GIRASOL Y MAIZ,7773103000002,FINO LIGHT ACEITE X 1.8LT,60.4,38.906897,2025-02-03,B1PB,397441SCN,2.0,88.80
1,ABARROTES,ABARROTES,ACEITES VEGETALES,ACEITES DE GIRASOL Y MAIZ,ACEITES DE GIRASOL Y MAIZ,7773103000002,FINO LIGHT ACEITE X 1.8LT,60.4,38.906897,2025-02-04,B1AJ,415847SCY,1.0,44.40
2,ABARROTES,ABARROTES,ACEITES VEGETALES,ACEITES DE GIRASOL Y MAIZ,ACEITES DE GIRASOL Y MAIZ,7773103000002,FINO LIGHT ACEITE X 1.8LT,60.4,38.906897,2025-02-12,B1BM,236648SES,1.0,44.40
3,ABARROTES,ABARROTES,ACEITES VEGETALES,ACEITES DE GIRASOL Y MAIZ,ACEITES DE GIRASOL Y MAIZ,7773103000002,FINO LIGHT ACEITE X 1.8LT,60.4,38.906897,2025-02-15,B1AY,286337SEK,1.0,44.40
4,ABARROTES,ABARROTES,ACEITES VEGETALES,ACEITES DE GIRASOL Y MAIZ,ACEITES DE GIRASOL Y MAIZ,7773103000002,FINO LIGHT ACEITE X 1.8LT,60.4,38.906897,2025-02-21,B116,2636215SE,1.0,42.62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3512428,PERECEDEROS,PANADERIA ENVASADA,PASTELERIA ENVASADA,PASTELES Y QUEQUES,PASTELES Y QUEQUES,7772619001954,LA SUPREMA QUEQUE MARMOLADO X 34GR CHOCO/NAR,1.7,1.299954,2025-03-04,B233,152201CAB,1.0,1.30
3512429,PERECEDEROS,PANADERIA ENVASADA,PASTELERIA ENVASADA,PASTELES Y QUEQUES,PASTELES Y QUEQUES,7772619001954,LA SUPREMA QUEQUE MARMOLADO X 34GR CHOCO/NAR,1.7,1.299954,2025-03-04,B122,4297284SJ,1.0,1.25
3512430,PERECEDEROS,PANADERIA ENVASADA,PASTELERIA ENVASADA,PASTELES Y QUEQUES,PASTELES Y QUEQUES,7772619001954,LA SUPREMA QUEQUE MARMOLADO X 34GR CHOCO/NAR,1.7,1.299954,2025-03-04,B154,2254793SAS,2.0,2.50
3512431,PERECEDEROS,PANADERIA ENVASADA,PASTELERIA ENVASADA,PASTELES Y QUEQUES,PASTELES Y QUEQUES,7772619001954,LA SUPREMA QUEQUE MARMOLADO X 34GR CHOCO/NAR,1.7,1.299954,2025-03-04,B1BO,157584SEV,3.0,3.90


In [20]:
clusters_une

,BODEGA,UNE,BODEGA2,CIUDAD,REGION,FORMATO,NSE,CLUSTER,BEAUTY,HOSPITALARIA
1,B901,FARMACORP,PO-115 AV. LITORAL ESQ. CHAYANTA,POTOSI,ALTIPLANO,MEDIANA,B,MEDIANA-B,SI,SI
2,B803,FARMACORP,PA - AV. TAHUAMANU ESQ. 9 DE FEBRERO,PANDO,LLANO,GRANDE,B,GRANDE-B,SI,NO
3,B802,FARMACORP,PA - AV. ENRIQUE FERNANDEZ C.(P. G.BUSCH),PANDO,LLANO,GRANDE,B,GRANDE-B,SI,SI
4,B801,FARMACORP,PA-83 AV.09 DE FEBRERO (DIAG C/BENI),PANDO,LLANO,PERMITIDO,B,PERMITIDO-B,SI,NO
5,B703,FARMACORP,BE-145 AV BOLIVAR ESQ ADOLFO VELASCO,BENI,LLANO,MEDIANA,B,MEDIANA-B,NO,NO
...,...,...,...,...,...,...,...,...,...,...
244,B109,FARMACORP,SC-09 ARGOMOSA Esq.CHARCAS,SANTA CRUZ,LLANO,GRANDE,C,GRANDE-C,NO,NO
245,B108,FARMACORP,SC-07 IRALA #564,SANTA CRUZ,LLANO,MEDIANA,B,MEDIANA-B,NO,NO
246,B104,FARMACORP,SC-03 IRALA-Esq. MONS. SANTIESTEBAN,SANTA CRUZ,LLANO,MEDIANA,A,MEDIANA-A,SI,SI
247,B102,FARMACORP,SC-02 PIRAI Esq.2doANILLO,SANTA CRUZ,LLANO,GRANDE,A,GRANDE-A,SI,SI


In [21]:
#base sobre la cual se va a trabajar
ventas_une = pd.merge(
    ventas_sample,
    clusters_une,
    left_on='COD_BODEGA',
    right_on='BODEGA',
)

In [22]:
ventas_une.NUMERO_FACTURA.nunique()

1788255

Se emitieron 151.342 facturas con al menos 1 SKU correspondiente a ETICOS y OTC en FARMACORP los dias  01, 02, 03, 04 de Marzo 2025.

Se emitieron 96.063 facturas con al menos 1 SKU correspondiente a BEBIDAS(SOLO HIDRATANTES Y ENERGIZANTES) y OTC en FARMACORP los dias 01-04 de Marzo 2025.

### Calculamos medidas

In [23]:
#calculamos el total de facturas emitidas en el rango de fechas de los datos
total_fact = ventas_une['NUMERO_FACTURA'].nunique()
total_fact

1788255

In [24]:
#contamos facturas en las que aparece al menos 1 vez cada SKU
ventas_sku = ventas_une.groupby(
    ['COD_ARTICULO', 'ARTICULO']
    )['NUMERO_FACTURA'].nunique().reset_index().sort_values(by='NUMERO_FACTURA', ascending=False)

In [25]:
#calculamos el peso de cada SKU sobre el total de las facturas
ventas_sku['weight'] = (ventas_sku['NUMERO_FACTURA']/total_fact).round(8)

In [26]:
#seleccionamos las facturas que solo tienen 1 sku
facturas_unicas = (
    ventas_une
    .groupby(['NUMERO_FACTURA'])
    .size()
    .loc[lambda x: x == 1]
    .reset_index()[['NUMERO_FACTURA']]
)

#filtramos la data con los sku que aparecen solos en 1 factura
ventas_solo = ventas_une.merge(
    facturas_unicas,
    on=['NUMERO_FACTURA'],
    how='inner'
)

In [27]:
#contamos facturas con un solo sku
total_fact_solo = ventas_solo['NUMERO_FACTURA'].nunique()
total_fact_solo

953689

In [28]:
#calculamos el % que representa del total de facturas
fact_solo = total_fact_solo/total_fact
percent_fact_solo = round(fact_solo*100, 2)
print(f"Se emite un ", percent_fact_solo, " % de facturas con un solo SKU durante el periodo analizado")

Se emite un  53.33  % de facturas con un solo SKU durante el periodo analizado


Del total de facturas emitidas feriado de carnavales de Marzo del 2025 63.015 facturas fueron por solo 1 SKU perteneciente a las ctaegorias BEBIDAS y OTC esto representa el 65.5% de las ventas totales en FARMACORP solo ese dia.

In [29]:
#contamos cuantas veces aparece cada sku solo 
ventas_sku_solo = ventas_solo.groupby(
    ['COD_ARTICULO', 'ARTICULO']
    )['NUMERO_FACTURA'].count().reset_index().sort_values(by='NUMERO_FACTURA', ascending=False)

In [30]:
ventas_sku_solo

,COD_ARTICULO,ARTICULO,NUMERO_FACTURA
6373,7703763995486,ZOPICLONA 7.5MG X 30 TAB (LA SANTE),11114
7240,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,9851
1990,255701,MIGRANOL X 100 COMP V+,5941
3986,700607,COLECTOR DE ORINA TRANS X 120/100/80ML STERIL ...,4958
7838,7770108121671,TYPIREC X 200 CAP BLANDA (ANTIGRIPAL),4920
...,...,...,...
32,022796916020,OGX ACOND BRAZILIAN KERATIN X 385ML,1
34,022796916129,OGX ACOND X 385ML ARGAN OIL OF MOROCCO,1
39,026217222387,CROFT&BARROW BILLETERA CUERO IMAN CAFE OSCURO ...,1
53,03014260275143,GILLETTE MACH3 TURBO RPTO X 2 UNID<DESC>,1


El SKU que se vende mas solo fue QUETOROL, seguido de ZOPICLIONA, RESSAKA y VITAMINAS MULTISABOR

In [31]:
#calculamos el peso de cada sku en las facturas solas
ventas_sku_solo['weight'] = ventas_sku_solo['NUMERO_FACTURA']/total_fact_solo

In [32]:
ventas_sku

,COD_ARTICULO,ARTICULO,NUMERO_FACTURA,weight
8516,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,25898,1.448227e-02
6730,751353,VITAMINA C MULTISABOR 60MG X 320 COMP GENERICO LI,24185,1.352436e-02
7573,7703763995486,ZOPICLONA 7.5MG X 30 TAB (LA SANTE),23362,1.306413e-02
9133,7770108121671,TYPIREC X 200 CAP BLANDA (ANTIGRIPAL),19561,1.093860e-02
5569,745391,VITAMINA C MULTISABOR 100MG X 500 COMP GENERICO,16453,9.200590e-03
...,...,...,...,...
8356,7757310001694,PVM S/LACTOSA X 460GR FRESA<DESC>,1,5.600000e-07
4495,60076,CLINDACNE 1% LOCION X 30ML CLINDAMICINA<DESC>,1,5.600000e-07
4488,6001159119883,BIO-OIL GEL X 200ML PARA PIEL SECA<DESC>,1,5.600000e-07
4467,5902169037932,BIELENDA MAGIC BRONZE CR CORP X 200ML DARK SKI...,1,5.600000e-07


In [33]:
#unimos todo
skus_analysis = pd.merge(
    ventas_sku,
    ventas_sku_solo,
    on='COD_ARTICULO',
    how='left',
    suffixes=['_gral', '_alone']
)

In [34]:
#calculamos el % en peso de las veces que cada sku aparece solo vs el total de veces que aparece en una factura
skus_analysis['weight_overeach'] = skus_analysis['NUMERO_FACTURA_alone']/skus_analysis['NUMERO_FACTURA_gral']

In [35]:
skus_analysis['weight_overall'] = skus_analysis['NUMERO_FACTURA_alone']/total_fact

In [36]:
skus_analysis = skus_analysis[[
    'COD_ARTICULO', 'ARTICULO_gral', 
    'NUMERO_FACTURA_gral', 'NUMERO_FACTURA_alone', 
    'weight_gral','weight_alone',
    'weight_overeach', 'weight_overall'
]].copy()

NRO_FACTURAS_gral: número total de facturas en las que el SKU aparece al menos una vez, independientemente de si la factura contiene otros SKUs.

NRO_FACTURAS_alone: número de facturas en las que el SKU aparece de forma exclusiva, es decir, facturas que contienen únicamente ese SKU y ningún otro.

weight_gral: proporción de facturas en las que aparece el SKU respecto al total de facturas emitidas. Mide la presencia general del SKU en el conjunto completo de facturación.

weight_alone: proporción de facturas de un solo SKU en las que aparece el SKU, respecto al total de facturas que contienen únicamente un SKU. Refleja la participación del SKU dentro de las facturas unitarias.

weight_overeach: proporción de facturas en las que el SKU aparece solo respecto al total de facturas en las que dicho SKU aparece al menos una vez. Indica qué tan frecuentemente el SKU se vende de manera exclusiva cuando está presente en una factura.

weight_overall: proporción de facturas en las que el SKU aparece solo respecto al total de facturas emitidas. Representa el peso absoluto de las ventas exclusivas del SKU sobre toda la facturación.

In [38]:
#filtramos solo los sku que aparecen solos en al menos 100 facturas
skus_analysis = skus_analysis[skus_analysis['NUMERO_FACTURA_alone']>=50]

In [39]:
skus_analysis

,COD_ARTICULO,ARTICULO_gral,NUMERO_FACTURA_gral,NUMERO_FACTURA_alone,weight_gral,weight_alone,weight_overeach,weight_overall
0,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,25898,9851.0,0.014482,0.010329,0.380377,0.005509
1,751353,VITAMINA C MULTISABOR 60MG X 320 COMP GENERICO LI,24185,3061.0,0.013524,0.003210,0.126566,0.001712
2,7703763995486,ZOPICLONA 7.5MG X 30 TAB (LA SANTE),23362,11114.0,0.013064,0.011654,0.475730,0.006215
3,7770108121671,TYPIREC X 200 CAP BLANDA (ANTIGRIPAL),19561,4920.0,0.010939,0.005159,0.251521,0.002751
4,745391,VITAMINA C MULTISABOR 100MG X 500 COMP GENERICO,16453,2222.0,0.009201,0.002330,0.135051,0.001243
...,...,...,...,...,...,...,...,...
5692,7800007791894,SENTIS 18.75MG X 30 CAP FENTERMINA (PSICO),72,60.0,0.000040,0.000063,0.833333,0.000034
5716,COMUNICATE-10,COMUNICATE 10 BS TARJETA TELEFONO,71,51.0,0.000040,0.000053,0.718310,0.000029
5725,7800026993354,SAMEXID 50MG X 30 CAP LISDEXANFETAMINA (PSICO),71,59.0,0.000040,0.000062,0.830986,0.000033
5789,7771011350233,LEVICALM 100MG/ML SOL ORAL X 150ML LEVETIRACETAM,70,54.0,0.000039,0.000057,0.771429,0.000030


In [40]:
skus_analysis = skus_analysis.sort_values(
    by=['weight_gral'], 
    ascending=[False]).round(5)

In [ ]:
#skus_analysis.to_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\output\analysis_sku.xlsx", index=False)

### Searching the AyB best pairs for top 500

In [48]:
# selfmerge for finding pairs of CAT 4 on same invoice
pairs_factura = pd.merge(
    ventas_une,
    ventas_une,
    on=["NUMERO_FACTURA"],
    suffixes=("_A", "_B")
)

In [49]:
# filtering combinations to avoid duplicates
pairs_factura = pairs_factura[pairs_factura["ARTICULO_A"] != pairs_factura["ARTICULO_B"]]

# keeping only one order of each pair
pairs_factura = pairs_factura[pairs_factura["ARTICULO_A"] > pairs_factura["ARTICULO_B"]]

# keeping only one order of each pair
pairs_factura = pairs_factura[pairs_factura["CAT 0_A"] != pairs_factura["CAT 0_B"]]


In [50]:
ventasABxfac = pairs_factura.groupby([
    'CAT 0_A', 'CAT 1_A', 'CAT 2_A', 'CAT 3_A', 'CAT 4_A', 'COD_ARTICULO_A', 'ARTICULO_A',
    'CAT 0_B', 'CAT 1_B', 'CAT 2_B', 'CAT 3_B', 'CAT 4_B', 'COD_ARTICULO_B', 'ARTICULO_B'
]
    ).agg({'NUMERO_FACTURA':'nunique'}).reset_index()

In [51]:
ventasABxfac = ventasABxfac.rename(columns={'NUMERO_FACTURA' : 'NUMERO_FACTURA_AB'})

In [52]:
ventasABxfac.sort_values(by=['NUMERO_FACTURA_AB'], ascending=[False])

,CAT 0_A,CAT 1_A,CAT 2_A,CAT 3_A,CAT 4_A,COD_ARTICULO_A,ARTICULO_A,CAT 0_B,CAT 1_B,CAT 2_B,CAT 3_B,CAT 4_B,COD_ARTICULO_B,ARTICULO_B,NUMERO_FACTURA_AB
777295,INSUMOS MEDICOS,INSUMOS MEDICOS,OTROS INSUMOS MEDICOS,INSUMOS DESCARTABLES,INSUMOS DESCARATABLES DE USO GENERAL,106421,JERINGA DESC 10ML C/AG 21 X 1 1/2 FARMACORP<H...,ETICOS,ETICOS AGUDOS,HORMONAS,CORTICOSTEROIDES VIA GENERAL(CORTICOIDES),CORTICOSTEROIDES SOLOS,751140,DEXAMETASONA 8MG IM-IV X 100 AMP/2ML GENERICO LI,1960
777170,INSUMOS MEDICOS,INSUMOS MEDICOS,OTROS INSUMOS MEDICOS,INSUMOS DESCARTABLES,INSUMOS DESCARATABLES DE USO GENERAL,106421,JERINGA DESC 10ML C/AG 21 X 1 1/2 FARMACORP<H...,ETICOS,ETICOS AGUDOS,APARATO LOCOMOTOR,ANTIINFLAMATORIOS Y ANTIRREUMATICOS,ANTIRREUMATICOS NO ESTEROIDEOS,751145,DICLOFENACO 75MG IM X 50 AMP GENERICO LI,1377
1046873,OTC,VITAMINAS Y MINERALES,VITAMINAS,VITAMINA C,VITAMINA C,751304,VITAMINA C 1GR IV X 50 AMP 5ML/2ML GENERICO LI,INSUMOS MEDICOS,INSUMOS MEDICOS,OTROS INSUMOS MEDICOS,INSUMOS DESCARTABLES,INSUMOS DESCARTABLES HOSPITALARIOS,709726,EQUIPO P/SUERO EN Y VENOCLISI C/AGUJA DOBLE PU...,1298
778268,INSUMOS MEDICOS,INSUMOS MEDICOS,OTROS INSUMOS MEDICOS,INSUMOS DESCARTABLES,INSUMOS DESCARATABLES DE USO GENERAL,106421,JERINGA DESC 10ML C/AG 21 X 1 1/2 FARMACORP<H...,OTC,VITAMINAS Y MINERALES,COMPLEJO B,COMPLEJO B AMPOLLA,COMPLEJO B AMPOLLA,751060,COMPLEJO B IM-IV X 100 AMP GENERICO LI,1115
778345,INSUMOS MEDICOS,INSUMOS MEDICOS,OTROS INSUMOS MEDICOS,INSUMOS DESCARTABLES,INSUMOS DESCARATABLES DE USO GENERAL,106421,JERINGA DESC 10ML C/AG 21 X 1 1/2 FARMACORP<H...,OTC,VITAMINAS Y MINERALES,VITAMINAS,B1 ASOCIAC. B6 Y/O B12,B1 ASOCIAC. B6 Y/O B12,120531,COBA VIMIN CTO 25000 IM X 25 AMP VIT B1 B6 B12,966
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14,ABARROTES,ABARROTES,ACEITES VEGETALES,ACEITES DE GIRASOL Y MAIZ,ACEITES DE GIRASOL Y MAIZ,7773103000064,FINO VEGETAL ACEITE X 900ML,BEBIDAS,AGUAS-ISOTONICOS Y ENERGIZANTES,AGUA,AGUA SIN GAS,AGUA SIN GAS,7771609001677,AGUA VITAL S/GAS X 3LT (PQT X 6U),1
15,ABARROTES,ABARROTES,ACEITES VEGETALES,ACEITES DE GIRASOL Y MAIZ,ACEITES DE GIRASOL Y MAIZ,7773103000064,FINO VEGETAL ACEITE X 900ML,BEBIDAS,GASEOSAS Y JUGOS,GASEOSAS,GASEOSAS,GASEOSAS,7771609000960,COCA COLA X 3LT (PQT X 6U),1
189128,CUIDADO PERSONAL,CUIDADO PERSONAL VARIOS,CUIDADO BUCAL,ORTODONCIA,CERA ORTODONCIA,8414600445536,FORAMEN CERA DE ORTODONCIA PQTE X 5 UNID,OTC,GASTRICO,APARATO DIGESTIVO Y METABOLICO,ANTIDIARREICOS,MICROORGANISMOS ANTIDIARREICOS,7770101006807,FLORESTOR POLVO 250MG X 10 SOBRES SACCHAROMYCES,1
17,ABARROTES,ABARROTES,ACEITES VEGETALES,ACEITES DE GIRASOL Y MAIZ,ACEITES DE GIRASOL Y MAIZ,7773103000064,FINO VEGETAL ACEITE X 900ML,BEBIDAS,GASEOSAS Y JUGOS,GASEOSAS,GASEOSAS LIGHT Y ZERO,GASEOSAS LIGHT Y ZERO,7771609001998,COCA COLA SIN AZUCAR X 3LT (PQT X 6U),1


In [ ]:
ventasABxfac['CAT 0_A', 'CAT 0_B', 'ARTICULO_A', 'ARTICULO_B', 'NUMERO_FACTURA_AB']

In [53]:
ventasABxfac['CAT 2_A'].value_counts()

CAT 2_A
APARATO DIGESTIVO Y METABOLICO    79965
SISTEMA NERVIOSO CENTRAL          72666
APARATO RESPIRATORIO              69456
GOLOSINAS                         44340
HIDRATANTES                       41643
                                  ...  
HARINAS                              26
CONDIMENTOS                          17
NAVIDAD                              10
CARNES                                5
VINAGRES                              4
Name: count, Length: 147, dtype: int64

In [54]:
ventasABxfac['weight_AB'] = ventasABxfac['NUMERO_FACTURA_AB'] / total_fact

In [55]:
ventasxpairs = pd.merge(
    ventasABxfac,
    ventas_sku,
    left_on='COD_ARTICULO_A',
    right_on='COD_ARTICULO'
)

In [ ]:
ventasxpairs_final = pd.merge(
    ventasxpairs[[
        'CAT 0_A', 'CAT 1_A', 'CAT 2_A', 'CAT 3_A', 'CAT 4_A', 'COD_ARTICULO_A', 'ARTICULO_A', 
        'CAT 0_B', 'CAT 1_B', 'CAT 2_B', 'CAT 3_B', 'CAT 4_B', 'COD_ARTICULO_B', 'ARTICULO_B', 
        'NUMERO_FACTURA_AB', 'weight_AB', 'NUMERO_FACTURA', 'weight']],
    ventas_sku,
    left_on='COD_ARTICULO_B',
    right_on='COD_ARTICULO'
)

In [ ]:
ventasxpairs_final = ventasxpairs_final.rename(columns={'weight_x':'weight_A',
                                                        'NUMERO_FACTURA_x':'NUMERO_FACTURA_A',
                                                        'weight_y':'weight_B',
                                                        'NUMERO_FACTURA_y':'NUMERO_FACTURA_B'})

In [ ]:
ventasxpairs_final = ventasxpairs_final[[
    'CAT 0_A', 'CAT 1_A', 'CAT 2_A', 'CAT 3_A', 'CAT 4_A', 'COD_ARTICULO_A', 'ARTICULO_A', 
    'CAT 0_B', 'CAT 1_B', 'CAT 2_B', 'CAT 3_B', 'CAT 4_B', 'COD_ARTICULO_B', 'ARTICULO_B', 
    'NUMERO_FACTURA_A', 'weight_A', 
    'NUMERO_FACTURA_B', 'weight_B',
    'NUMERO_FACTURA_AB', 'weight_AB'
]]

In [ ]:
ventasxpairs_final = ventasxpairs_final.sort_values(
    by=['COD_ARTICULO_A',
        'NUMERO_FACTURA_AB','weight_AB'], 
        ascending=[False, False, False])

In [ ]:
ventasxpairs_final

In [ ]:
top_sku_pairs = ventasxpairs_final.groupby('COD_ARTICULO_A').head(10).reset_index(drop=True)

### Getting the final base for analysis

In [ ]:
base_analysis = pd.merge(
    top_sku_pairs,
    skus_analysis,
    left_on='COD_ARTICULO_A',
    right_on='COD_ARTICULO',
    how='right')

In [ ]:
base = pd.merge(
    base_analysis,
    cubo[['COD_ARTICULO', 'Precio Unitario FA', 'CR Unitario']],
    left_on='COD_ARTICULO_B',
    right_on='COD_ARTICULO',
    how='left'
)

In [ ]:
base_cat1 = pd.merge(
    base,
    cubo[['COD_ARTICULO', 'CAT 1']],
    left_on='COD_ARTICULO_A',
    right_on='COD_ARTICULO',
    how='left'
)

In [ ]:
base_cat1

In [ ]:
base_cat1[['CAT 2_A', 'CAT 2_B']].value_counts()

In [ ]:
base_final = base_cat1[[
    'CAT 0_A', 'CAT 1_A', 'CAT 2_A', 'CAT 3_A', 'CAT 4_A',
    'COD_ARTICULO_A', 'ARTICULO_A', 

    'CAT 0_B','CAT 1_B', 'CAT 2_B', 'CAT 3_B', 'CAT 4_B',
    'COD_ARTICULO_B', 'ARTICULO_B',
    
    'NUMERO_FACTURA_alone', 'NUMERO_FACTURA_A', 'NUMERO_FACTURA_B', 'NUMERO_FACTURA_AB',
    #'weight_A', 'weight_B', 'weight_AB', 'weight_alone', 'weight_overeach', 'weight_overall',
    
    'Precio Unitario FA',
    'CR Unitario'
]]

In [ ]:
base_final = base_final.rename(columns={
    'Precio Unitario FA': 'PVU_B',
    'CR Unitario' : 'CRU_B'
    })

In [ ]:
base_final['Profit_B'] = base_final['PVU_B'] - base_final['CRU_B']

In [ ]:
base_final

In [ ]:
profit_top_sku = (
    base_final
    .groupby('COD_ARTICULO_A', as_index=False)['Profit_B']
    .mean()
    .rename(columns={'Profit_B': 'Profit_B_avg'})
)

In [ ]:
base_final = pd.merge(
    base_final,
    profit_top_sku,
    on='COD_ARTICULO_A'
)

In [ ]:
base_final['ambition'] = 0.05

In [ ]:
base_final['opportunity'] = base_final['ambition'] * base_final['NUMERO_FACTURA_alone'] * base_final['Profit_B']

In [ ]:
base_final['opportunity_avg'] = base_final['ambition'] * base_final['NUMERO_FACTURA_alone'] * base_final['Profit_B_avg']

In [ ]:
base_final

In [ ]:
base_final.to_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\cross selling\output\cross_selling_AMARKET.xlsx",
                     index=False)

In [ ]:
# Top 50 ARTICULO_A by NRO_FACTURAS_A
if 'NUMERO_FACTURA_A' not in base.columns:
    raise KeyError("'NUMERO_FACTURA_A' not found in `base`. Check previous merges.")

group_cols = ['COD_ARTICULO_A', 'ARTICULO_A']

top50_articulo_A = (
    base
    .groupby(group_cols, as_index=False)['NUMERO_FACTURA_A']
    .max()
    .sort_values(by='NUMERO_FACTURA_A', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

# attach weight_A if available
if 'weight_A' in base.columns:
    top50_articulo_A = top50_articulo_A.merge(
        base.groupby(group_cols, as_index=False)['weight_A'].max(),
        on=group_cols,
        how='left'
    )


In [ ]:
# display
top50_articulo_A

Los articulos que son generadores de volumen de compra son: RESAKKA, QUETOROL, GLUCOSAMIN, TYPIREC, VITAMINAS C MULTISABOR (60 Y 100 MG), ZOPLICONA, DIGESTAN, MIGRANOL, VIADIL, ALIA2

In [ ]:
# Top 50 ARTICULO_B by NRO_FACTURAS_B (including partners count)
if 'NUMERO_FACTURA_B' not in base.columns:
    raise KeyError("'NUMERO_FACTURA_B' not found in `base`. Check previous merges.")

group_cols_b = ['COD_ARTICULO_B', 'ARTICULO_B']

# aggregate invoices per ARTICULO_B
agg_b = (
    base
    .groupby(group_cols_b, as_index=False)
    .agg(NUMERO_FACTURA_B=('NUMERO_FACTURA_B', 'max'))
)

# count distinct partners (COD_ARTICULO_A) per ARTICULO_B
partners = (
    base
    .groupby(group_cols_b, as_index=False)
    .agg(partners_count=('COD_ARTICULO_A', 'nunique'))
)

# merge results
top50_articulo_B = agg_b.merge(partners, on=group_cols_b, how='left')

# attach weight_B if available
if 'weight_B' in base.columns:
    w = base.groupby(group_cols_b, as_index=False)['weight_B'].max()
    top50_articulo_B = top50_articulo_B.merge(w, on=group_cols_b, how='left')

# sort and take top 50
top50_articulo_B = (
    top50_articulo_B
    .sort_values(by='NUMERO_FACTURA_B', ascending=False)
    .head(10)
    .reset_index(drop=True)
)


In [ ]:
# display
top50_articulo_B